<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **SpaceX  Falcon 9 first stage Landing Prediction**


# Lab 1: Collecting the data


Estimated time needed: **45** minutes


In this capstone, we will predict if the Falcon 9 first stage will land successfully. SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars; other providers cost upward of 165 million dollars each, much of the savings is because SpaceX can reuse the first stage. Therefore if we can determine if the first stage will land, we can determine the cost of a launch. This information can be used if an alternate company wants to bid against SpaceX for a rocket launch. In this lab, you will collect and make sure the data is in the correct format from an API. The following is an example of a successful and launch.


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/crash.gif)


Most unsuccessful landings are planned. Space X performs a controlled landing in the oceans. 


## Objectives


In this lab, you will make a get request to the SpaceX API. You will also do some basic data wrangling and formating. 

- Request to the SpaceX API
- Clean the requested data


----


## Import Libraries and Define Auxiliary Functions


We will import the following libraries into the lab


In [1]:
# Requests allows us to make HTTP requests which we will use to get data from an API
import requests
# Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
# NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays
import numpy as np
# Datetime is a library that allows us to represent dates
import datetime

# Setting this option will print all collumns of a dataframe
pd.set_option('display.max_columns', None)
# Setting this option will print all of the data in a feature
pd.set_option('display.max_colwidth', None)

Below we will define a series of helper functions that will help us use the API to extract information using identification numbers in the launch data.

From the <code>rocket</code> column we would like to learn the booster name.


In [2]:
# Takes the dataset and uses the rocket column to call the API and append the data to the list
def getBoosterVersion(data):
    for x in data['rocket']:
       if x:
        response = requests.get("https://api.spacexdata.com/v4/rockets/"+str(x)).json()
        BoosterVersion.append(response['name'])

From the <code>launchpad</code> we would like to know the name of the launch site being used, the logitude, and the latitude.


In [3]:
# Takes the dataset and uses the launchpad column to call the API and append the data to the list
def getLaunchSite(data):
    for x in data['launchpad']:
       if x:
         response = requests.get("https://api.spacexdata.com/v4/launchpads/"+str(x)).json()
         Longitude.append(response['longitude'])
         Latitude.append(response['latitude'])
         LaunchSite.append(response['name'])

From the <code>payload</code> we would like to learn the mass of the payload and the orbit that it is going to.


In [4]:
# Takes the dataset and uses the payloads column to call the API and append the data to the lists
def getPayloadData(data):
    for load in data['payloads']:
       if load:
        response = requests.get("https://api.spacexdata.com/v4/payloads/"+load).json()
        PayloadMass.append(response['mass_kg'])
        Orbit.append(response['orbit'])

From <code>cores</code> we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, wheter the core is reused, wheter legs were used, the landing pad used, the block of the core which is a number used to seperate version of cores, the number of times this specific core has been reused, and the serial of the core.


In [5]:
# Takes the dataset and uses the cores column to call the API and append the data to the lists
def getCoreData(data):
    for core in data['cores']:
            if core['core'] != None:
                response = requests.get("https://api.spacexdata.com/v4/cores/"+core['core']).json()
                Block.append(response['block'])
                ReusedCount.append(response['reuse_count'])
                Serial.append(response['serial'])
            else:
                Block.append(None)
                ReusedCount.append(None)
                Serial.append(None)
            Outcome.append(str(core['landing_success'])+' '+str(core['landing_type']))
            Flights.append(core['flight'])
            GridFins.append(core['gridfins'])
            Reused.append(core['reused'])
            Legs.append(core['legs'])
            LandingPad.append(core['landpad'])

Now let's start requesting rocket launch data from SpaceX API with the following URL:


In [6]:
spacex_url="https://api.spacexdata.com/v4/launches/past"

In [7]:
response = requests.get(spacex_url)

Check the content of the response


In [8]:
print(response.content)

b'<!DOCTYPE html>\n<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->\n<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->\n<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->\n<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->\n<head>\n\n<title>spacexdata.com | 525: SSL handshake failed</title>\n<meta charset="UTF-8" />\n<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />\n<meta http-equiv="X-UA-Compatible" content="IE=Edge" />\n<meta name="robots" content="noindex, nofollow" />\n<meta name="viewport" content="width=device-width,initial-scale=1" />\n<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/main.css" />\n</head>\n<body>\n<div id="cf-wrapper">\n    <div id="cf-error-details" class="p-0">\n        <header class="mx-auto pt-10 lg:pt-6 lg:px-8 w-240 lg:w-full mb-8">\n            <h1 class="inline-block sm:block sm:mb-2 font-light text-60 lg:text-4xl text-bla

You should see the response contains massive information about SpaceX launches. Next, let's try to discover some more relevant information for this project.


### Task 1: Request and parse the SpaceX launch data using the GET request


To make the requested JSON results more consistent, we will use the following static response object for this project:


In [9]:
static_json_url='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'

We should see that the request was successfull with the 200 status response code


In [10]:
response=requests.get(static_json_url)

In [11]:
response.status_code

200

Now we decode the response content as a Json using <code>.json()</code> and turn it into a Pandas dataframe using <code>.json_normalize()</code>


In [12]:
# Use json_normalize meethod to convert the json result into a dataframe
data_json = response.json()
data = pd.json_normalize(data_json)

Using the dataframe <code>data</code> print the first 5 rows


In [13]:
# Get the head of the dataframe
print("\nFirst 5 rows of the dataset:")
print(data.head())

print(f"\nDataset shape: {data.shape}")
print(f"\nColumns: {list(data.columns)}")


First 5 rows of the dataset:
       static_fire_date_utc  static_fire_date_unix    tbd    net  window  \
0  2006-03-17T00:00:00.000Z           1.142554e+09  False  False     0.0   
1                      None                    NaN  False  False     0.0   
2                      None                    NaN  False  False     0.0   
3  2008-09-20T00:00:00.000Z           1.221869e+09  False  False     0.0   
4                      None                    NaN  False  False     0.0   

                     rocket  success  \
0  5e9d0d95eda69955f709d1eb    False   
1  5e9d0d95eda69955f709d1eb    False   
2  5e9d0d95eda69955f709d1eb    False   
3  5e9d0d95eda69955f709d1eb     True   
4  5e9d0d95eda69955f709d1eb     True   

                                                                                                                                                                                details  \
0                                                                                    

You will notice that a lot of the data are IDs. For example the rocket column has no information about the rocket just an identification number.

We will now use the API again to get information about the launches using the IDs given for each launch. Specifically we will be using columns <code>rocket</code>, <code>payloads</code>, <code>launchpad</code>, and <code>cores</code>.


In [14]:
# Lets take a subset of our dataframe keeping only the features we want and the flight number, and date_utc.
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]

# We will remove rows with multiple cores because those are falcon rockets with 2 extra rocket boosters and rows that have multiple payloads in a single rocket.
data = data[data['cores'].map(len)==1]
data = data[data['payloads'].map(len)==1]

# Since payloads and cores are lists of size 1 we will also extract the single value in the list and replace the feature.
data['cores'] = data['cores'].map(lambda x : x[0])
data['payloads'] = data['payloads'].map(lambda x : x[0])

# We also want to convert the date_utc to a datetime datatype and then extracting the date leaving the time
data['date'] = pd.to_datetime(data['date_utc']).dt.date

# Using the date we will restrict the dates of the launches
data = data[data['date'] <= datetime.date(2020, 11, 13)]

* From the <code>rocket</code> we would like to learn the booster name

* From the <code>payload</code> we would like to learn the mass of the payload and the orbit that it is going to

* From the <code>launchpad</code> we would like to know the name of the launch site being used, the longitude, and the latitude.

* **From <code>cores</code> we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, whether the core is reused, whether legs were used, the landing pad used, the block of the core which is a number used to seperate version of cores, the number of times this specific core has been reused, and the serial of the core.**

The data from these requests will be stored in lists and will be used to create a new dataframe.


In [21]:
#Global variables 
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []


def safe_api_get(url):
    """
    Safely make API request with retry logic.
    Returns JSON dict or None if failed.
    """
    max_retries = 3
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, timeout=10)
            if resp.status_code == 200:
                return resp.json()
            else:
                print(f"  Status {resp.status_code} for {url}")
                return None
        except Exception as e:
            print(f"  Attempt {attempt+1} failed for {url}: {e}")
            time.sleep(1)
    return None


def getBoosterVersion(data):
    """Fetch rocket name from API using rocket ID."""
    global BoosterVersion
    BoosterVersion = []
    
    for x in data['rocket']:
        if pd.isna(x) or x == '':
            BoosterVersion.append(np.nan)
            continue
            
        response = safe_api_get(f"https://api.spacexdata.com/v4/rockets/{str(x)}")
        
        if response and 'name' in response:
            BoosterVersion.append(response['name'])
        else:
            BoosterVersion.append(np.nan)
            print(f"  Could not get rocket data for ID: {x}")


def getLaunchSite(data):
    """Fetch launch site info from API using launchpad ID."""
    global LaunchSite, Longitude, Latitude
    LaunchSite = []
    Longitude = []
    Latitude = []
    
    for x in data['launchpad']:
        if pd.isna(x) or x == '':
            LaunchSite.append(np.nan)
            Longitude.append(np.nan)
            Latitude.append(np.nan)
            continue
            
        response = safe_api_get(f"https://api.spacexdata.com/v4/launchpads/{str(x)}")
        
        if response:
            LaunchSite.append(response.get('name', np.nan))
            Longitude.append(response.get('longitude', np.nan))
            Latitude.append(response.get('latitude', np.nan))
        else:
            LaunchSite.append(np.nan)
            Longitude.append(np.nan)
            Latitude.append(np.nan)
            print(f"  Could not get launchpad data for ID: {x}")


def getPayloadData(data):
    """Fetch payload mass and orbit from API using payload ID."""
    global PayloadMass, Orbit
    PayloadMass = []
    Orbit = []
    
    for x in data['payloads']:
        if pd.isna(x) or x == '':
            PayloadMass.append(np.nan)
            Orbit.append(np.nan)
            continue
            
        response = safe_api_get(f"https://api.spacexdata.com/v4/payloads/{str(x)}")
        
        if response:
            PayloadMass.append(response.get('mass_kg', np.nan))
            Orbit.append(response.get('orbit', np.nan))
        else:
            PayloadMass.append(np.nan)
            Orbit.append(np.nan)
            print(f"  Could not get payload data for ID: {x}")


def getCoreData(data):
    """Fetch core data from API using core ID."""
    global Outcome, Flights, GridFins, Reused, Legs, LandingPad, Block, ReusedCount, Serial
    Outcome = []
    Flights = []
    GridFins = []
    Reused = []
    Legs = []
    LandingPad = []
    Block = []
    ReusedCount = []
    Serial = []
    
    for core_dict in data['cores']:
        if not isinstance(core_dict, dict):
            _append_core_nans()
            continue
        
        core_id = core_dict.get('core', '')
        
        # Get core details from API
        if core_id and not pd.isna(core_id):
            core_response = safe_api_get(f"https://api.spacexdata.com/v4/cores/{str(core_id)}")
        else:
            core_response = None
        
        # Landing outcome
        landing_success = core_dict.get('landing_success')
        landing_type = core_dict.get('landing_type', '')
        
        if landing_success is not None:
            Outcome.append(f"{landing_success} {landing_type}".strip())
        else:
            Outcome.append(np.nan)
        
        Flights.append(core_dict.get('flight', np.nan))
        GridFins.append(core_dict.get('gridfins', np.nan))
        Reused.append(core_dict.get('reused', np.nan))
        Legs.append(core_dict.get('legs', np.nan))
        LandingPad.append(core_dict.get('landpad', np.nan))
        
        if core_response:
            Block.append(core_response.get('block', np.nan))
            ReusedCount.append(core_response.get('reuse_count', np.nan))
            Serial.append(core_response.get('serial', np.nan))
        else:
            Block.append(np.nan)
            ReusedCount.append(np.nan)
            Serial.append(np.nan)
            if core_id:
                print(f"  Could not get core data for ID: {core_id}")


def _append_core_nans():
    """Helper to append NaN for all core lists."""
    Outcome.append(np.nan)
    Flights.append(np.nan)
    GridFins.append(np.nan)
    Reused.append(np.nan)
    Legs.append(np.nan)
    LandingPad.append(np.nan)
    Block.append(np.nan)
    ReusedCount.append(np.nan)
    Serial.append(np.nan)


# ============================================================
# STEP 4: Run the functions
# ============================================================

print("\n" + "="*50)
print("Fetching data from SpaceX API...")
print("="*50)



Fetching data from SpaceX API...


These functions will apply the outputs globally to the above variables. Let's take a looks at <code>BoosterVersion</code> variable. Before we apply  <code>getBoosterVersion</code> the list is empty:


In [22]:
BoosterVersion

[]

Now, let's apply <code> getBoosterVersion</code> function method to get the booster version


In [23]:
# Call getBoosterVersion
getBoosterVersion(data)

  Status 525 for https://api.spacexdata.com/v4/rockets/5e9d0d95eda69955f709d1eb
  Could not get rocket data for ID: 5e9d0d95eda69955f709d1eb
  Status 525 for https://api.spacexdata.com/v4/rockets/5e9d0d95eda69955f709d1eb
  Could not get rocket data for ID: 5e9d0d95eda69955f709d1eb
  Status 525 for https://api.spacexdata.com/v4/rockets/5e9d0d95eda69955f709d1eb
  Could not get rocket data for ID: 5e9d0d95eda69955f709d1eb
  Status 525 for https://api.spacexdata.com/v4/rockets/5e9d0d95eda69955f709d1eb
  Could not get rocket data for ID: 5e9d0d95eda69955f709d1eb
  Status 525 for https://api.spacexdata.com/v4/rockets/5e9d0d95eda69973a809d1ec
  Could not get rocket data for ID: 5e9d0d95eda69973a809d1ec
  Status 525 for https://api.spacexdata.com/v4/rockets/5e9d0d95eda69973a809d1ec
  Could not get rocket data for ID: 5e9d0d95eda69973a809d1ec
  Status 525 for https://api.spacexdata.com/v4/rockets/5e9d0d95eda69973a809d1ec
  Could not get rocket data for ID: 5e9d0d95eda69973a809d1ec
  Status 525 

the list has now been update 


In [24]:
BoosterVersion[0:5]

[nan, nan, nan, nan, nan]

we can apply the rest of the  functions here:


In [25]:
# Call getLaunchSite
getLaunchSite(data)

  Status 525 for https://api.spacexdata.com/v4/launchpads/5e9e4502f5090995de566f86
  Could not get launchpad data for ID: 5e9e4502f5090995de566f86
  Status 525 for https://api.spacexdata.com/v4/launchpads/5e9e4502f5090995de566f86
  Could not get launchpad data for ID: 5e9e4502f5090995de566f86
  Status 525 for https://api.spacexdata.com/v4/launchpads/5e9e4502f5090995de566f86
  Could not get launchpad data for ID: 5e9e4502f5090995de566f86
  Status 525 for https://api.spacexdata.com/v4/launchpads/5e9e4502f5090995de566f86
  Could not get launchpad data for ID: 5e9e4502f5090995de566f86
  Status 525 for https://api.spacexdata.com/v4/launchpads/5e9e4501f509094ba4566f84
  Could not get launchpad data for ID: 5e9e4501f509094ba4566f84
  Status 525 for https://api.spacexdata.com/v4/launchpads/5e9e4501f509094ba4566f84
  Could not get launchpad data for ID: 5e9e4501f509094ba4566f84
  Status 525 for https://api.spacexdata.com/v4/launchpads/5e9e4501f509094ba4566f84
  Could not get launchpad data for 

In [26]:
# Call getPayloadData
getPayloadData(data)

  Status 525 for https://api.spacexdata.com/v4/payloads/5eb0e4b5b6c3bb0006eeb1e1
  Could not get payload data for ID: 5eb0e4b5b6c3bb0006eeb1e1
  Status 525 for https://api.spacexdata.com/v4/payloads/5eb0e4b6b6c3bb0006eeb1e2
  Could not get payload data for ID: 5eb0e4b6b6c3bb0006eeb1e2
  Status 525 for https://api.spacexdata.com/v4/payloads/5eb0e4b7b6c3bb0006eeb1e5
  Could not get payload data for ID: 5eb0e4b7b6c3bb0006eeb1e5
  Status 525 for https://api.spacexdata.com/v4/payloads/5eb0e4b7b6c3bb0006eeb1e6
  Could not get payload data for ID: 5eb0e4b7b6c3bb0006eeb1e6
  Status 525 for https://api.spacexdata.com/v4/payloads/5eb0e4b7b6c3bb0006eeb1e7
  Could not get payload data for ID: 5eb0e4b7b6c3bb0006eeb1e7
  Status 525 for https://api.spacexdata.com/v4/payloads/5eb0e4bab6c3bb0006eeb1ea
  Could not get payload data for ID: 5eb0e4bab6c3bb0006eeb1ea
  Status 525 for https://api.spacexdata.com/v4/payloads/5eb0e4bbb6c3bb0006eeb1ed
  Could not get payload data for ID: 5eb0e4bbb6c3bb0006eeb1ed

In [27]:
# Call getCoreData
getCoreData(data)

  Status 525 for https://api.spacexdata.com/v4/cores/5e9e289df35918033d3b2623
  Could not get core data for ID: 5e9e289df35918033d3b2623
  Status 525 for https://api.spacexdata.com/v4/cores/5e9e289ef35918416a3b2624
  Could not get core data for ID: 5e9e289ef35918416a3b2624
  Status 525 for https://api.spacexdata.com/v4/cores/5e9e289ef3591855dc3b2626
  Could not get core data for ID: 5e9e289ef3591855dc3b2626
  Status 525 for https://api.spacexdata.com/v4/cores/5e9e289ef359184f103b2627
  Could not get core data for ID: 5e9e289ef359184f103b2627
  Status 525 for https://api.spacexdata.com/v4/cores/5e9e289ef359185f2b3b2628
  Could not get core data for ID: 5e9e289ef359185f2b3b2628
  Status 525 for https://api.spacexdata.com/v4/cores/5e9e289ef35918f39c3b262a
  Could not get core data for ID: 5e9e289ef35918f39c3b262a
  Status 525 for https://api.spacexdata.com/v4/cores/5e9e289ff3591884e03b262c
  Could not get core data for ID: 5e9e289ff3591884e03b262c
  Status 525 for https://api.spacexdata.c

Finally lets construct our dataset using the data we have obtained. We we combine the columns into a dictionary.


In [28]:
launch_dict = {'FlightNumber': list(data['flight_number']),
'Date': list(data['date']),
'BoosterVersion':BoosterVersion,
'PayloadMass':PayloadMass,
'Orbit':Orbit,
'LaunchSite':LaunchSite,
'Outcome':Outcome,
'Flights':Flights,
'GridFins':GridFins,
'Reused':Reused,
'Legs':Legs,
'LandingPad':LandingPad,
'Block':Block,
'ReusedCount':ReusedCount,
'Serial':Serial,
'Longitude': Longitude,
'Latitude': Latitude}

Then, we need to create a Pandas data frame from the dictionary launch_dict.


In [29]:
# Create a data from launch_dict
launch_df = pd.DataFrame(launch_dict)

Show the summary of the dataframe


In [30]:
# Show the head of the dataframe
print(f"\nLaunch DataFrame shape: {launch_df.shape}")
print(launch_df.head())


Launch DataFrame shape: (94, 17)
   FlightNumber        Date  BoosterVersion  PayloadMass  Orbit  LaunchSite  \
0             1  2006-03-24             NaN          NaN    NaN         NaN   
1             2  2007-03-21             NaN          NaN    NaN         NaN   
2             4  2008-09-28             NaN          NaN    NaN         NaN   
3             5  2009-07-13             NaN          NaN    NaN         NaN   
4             6  2010-06-04             NaN          NaN    NaN         NaN   

  Outcome  Flights  GridFins  Reused   Legs LandingPad  Block  ReusedCount  \
0     NaN        1     False   False  False       None    NaN          NaN   
1     NaN        1     False   False  False       None    NaN          NaN   
2     NaN        1     False   False  False       None    NaN          NaN   
3     NaN        1     False   False  False       None    NaN          NaN   
4     NaN        1     False   False  False       None    NaN          NaN   

   Serial  Longitude  

### Task 2: Filter the dataframe to only include `Falcon 9` launches


Finally we will remove the Falcon 1 launches keeping only the Falcon 9 launches. Filter the data dataframe using the <code>BoosterVersion</code> column to only keep the Falcon 9 launches. Save the filtered data to a new dataframe called <code>data_falcon9</code>.


In [31]:
# Hint data['BoosterVersion']!='Falcon 1'
launch_df['BoosterVersion'].value_counts()
data_falcon9 = launch_df[launch_df['BoosterVersion'] != 'Falcon 1']

Now that we have removed some values we should reset the FlgihtNumber column


In [32]:
data_falcon9.loc[:,'FlightNumber'] = list(range(1, data_falcon9.shape[0]+1))
data_falcon9

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2006-03-24,NaN,NaN,NaN,NaN,NaN,1,False,False,False,None,NaN,NaN,NaN,NaN,NaN
1,2,2007-03-21,NaN,NaN,NaN,NaN,NaN,1,False,False,False,None,NaN,NaN,NaN,NaN,NaN
2,3,2008-09-28,NaN,NaN,NaN,NaN,NaN,1,False,False,False,None,NaN,NaN,NaN,NaN,NaN
3,4,2009-07-13,NaN,NaN,NaN,NaN,NaN,1,False,False,False,None,NaN,NaN,NaN,NaN,NaN
4,5,2010-06-04,NaN,NaN,NaN,NaN,NaN,1,False,False,False,None,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89,90,2020-09-03,NaN,NaN,NaN,NaN,True ASDS,2,True,True,True,5e9e3032383ecb6bb234e7ca,NaN,NaN,NaN,NaN,NaN
90,91,2020-10-06,NaN,NaN,NaN,NaN,True ASDS,3,True,True,True,5e9e3032383ecb6bb234e7ca,NaN,NaN,NaN,NaN,NaN
91,92,2020-10-18,NaN,NaN,NaN,NaN,True ASDS,6,True,True,True,5e9e3032383ecb6bb234e7ca,NaN,NaN,NaN,NaN,NaN
92,93,2020-10-24,NaN,NaN,NaN,NaN,True ASDS,3,True,True,True,5e9e3033383ecbb9e534e7cc,NaN,NaN,NaN,NaN,NaN


## Data Wrangling


We can see below that some of the rows are missing values in our dataset.


In [33]:
data_falcon9.isnull().sum()

FlightNumber       0
Date               0
BoosterVersion    94
PayloadMass       94
Orbit             94
LaunchSite        94
Outcome           25
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        30
Block             94
ReusedCount       94
Serial            94
Longitude         94
Latitude          94
dtype: int64

Before we can continue we must deal with these missing values. The <code>LandingPad</code> column will retain None values to represent when landing pads were not used.


### Task 3: Dealing with Missing Values


Calculate below the mean for the <code>PayloadMass</code> using the <code>.mean()</code>. Then use the mean and the <code>.replace()</code> function to replace `np.nan` values in the data with the mean you calculated.


In [37]:
# Calculate the mean value of PayloadMass column
payload_mass_mean = data_falcon9['PayloadMass'].mean()
# Replace the np.nan values with its mean value
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].replace(np.nan, payload_mass_mean)
# Verify nulls are replaced
data_falcon9.isnull().sum()

FlightNumber       0
Date               0
BoosterVersion    94
PayloadMass       94
Orbit             94
LaunchSite        94
Outcome           25
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        30
Block             94
ReusedCount       94
Serial            94
Longitude         94
Latitude          94
dtype: int64

In [38]:
# Save to CSV
data_falcon9.to_csv('dataset_part_1.csv', index=False)

You should see the number of missing values of the <code>PayLoadMass</code> change to zero.


Now we should have no missing values in our dataset except for in <code>LandingPad</code>.


We can now export it to a <b>CSV</b> for the next section,but to make the answers consistent, in the next lab we will provide data in a pre-selected date range. 


<code>data_falcon9.to_csv('dataset_part_1.csv', index=False)</code>


## Authors


<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> has a PhD in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD. 


<!--## Change Log
-->


<!--

|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2020-09-20|1.1|Joseph|get result each time you run|
|2020-09-20|1.1|Azim |Created Part 1 Lab using SpaceX API|
|2020-09-20|1.0|Joseph |Modified Multiple Areas|
-->


Copyright ©IBM Corporation. All rights reserved.
